# Combine Founder Level Datasets
This notebook merges the primary founder-level event dataset with aggregated education history, prior work experience (industry and seniority counts), raw position details prior to founding, and Pitchbook venture outcomes.

## Datasets Involved
1. **Founder Level Event Dataset**: Primary dataset identifying each founding event.
2. **Education History**: Individual degree attainment and field indicators.
3. **Prior Experience - Industry**: Years of pre-founding experience spent in specific industries.
4. **Prior Experience - Seniority**: Years of pre-founding experience spent at each seniority level (1-7).
5. **Prior Position Details**: Raw job titles, category, and parent industry NAICS description prior to founding.
6. **Pitchbook Venture Outcomes**: Matched venture characteristic indicators (collapsed at LinkedIn rcid level).

In [1]:
import os
import pandas as pd
import numpy as np

# Define directories
BASE_DIR = "/Users/milanmiric/Documents/Research/Active Papers/LinkedIn Entreprenurship & Experience [Paper 2]/Data Extraction and Analysis"
DATA_DIR = os.path.join(BASE_DIR, "D - Data/D2 - Datasets for Matching")
OUTPUT_DIR = os.path.join(BASE_DIR, "D - Data/D3 - Refined Datasets for Analysis")

# Input file paths
EVENTS_FILE = os.path.join(DATA_DIR, "Founder_Level_Event_Dataset.csv")
EDU_FILE = os.path.join(DATA_DIR, "Founder_Education_Aggregated.csv")
EXP_IND_FILE = os.path.join(DATA_DIR, "Founder_Prior_Experience_Industry.csv")
EXP_SEN_FILE = os.path.join(DATA_DIR, "Founder_Prior_Experience_Seniority.csv")
EXP_ROLE_FILE = os.path.join(DATA_DIR, "Founder_Prior_Experience_Roles.csv")
DETAILS_FILE = os.path.join(DATA_DIR, "Founder_Prior_Position_Details.csv")
PB_FILE = os.path.join(DATA_DIR, "Matched_PB_Founded_Venture_Data.csv")

# Output file path
FINAL_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "Founder_Level_Merged_Dataset.csv")

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
print("Loading base event dataset...")
if not os.path.exists(EVENTS_FILE):
    raise FileNotFoundError(f"Base event dataset not found at {EVENTS_FILE}")

df_events = pd.read_csv(EVENTS_FILE)
print(f"Loaded {len(df_events):,} event rows.")
df_events.head()

Loading base event dataset...


Loaded 1,239,185 event rows.


,user_id,founding_event_number,years_since_first_job,last_pre_founding_salary,last_firm_id,venture_position_id,venture_firm_id,venture_metro_area,venture_msa,venture_country,...,venture_duration_years,total_prior_positions,max_prior_seniority,prior_sen_pos_count_1,prior_sen_pos_count_2,prior_sen_pos_count_3,prior_sen_pos_count_4,prior_sen_pos_count_5,prior_sen_pos_count_6,prior_sen_pos_count_7
0,1047840,1,2.250513,62608.200,198410.0,-2848061458383499091,10420471.0,anaheim metropolitan area,Los Angeles-Long Beach-Santa Ana CA MSA,United States,...,2.168378,1,1,1,0,0,0,0,0,0
1,1047840,2,8.503765,130091.164,20937770.0,423286751770052948,93663298.0,san francisco metropolitan area,San Francisco-Oakland-Fremont CA MSA,United States,...,0.495551,9,5,1,1,3,3,1,0,0
2,1057900,1,1.251198,25018.734,1207525.0,-4055469569477305836,5397841.0,eugene metropolitan area,Corvalis OR MSA,United States,...,1.749487,6,5,4,1,0,0,1,0,0
3,1057900,2,3.252567,25721.246,210631.0,5527140130802678961,5397841.0,eugene metropolitan area,Corvalis OR MSA,United States,...,1.333333,7,5,4,2,0,0,1,0,0
4,1058400,1,23.832991,139438.520,441760.0,7469691805602090568,93652283.0,north carolina nonmetropolitan area,empty,United States,...,2.833676,9,6,1,2,0,2,3,1,0


In [3]:
if os.path.exists(EDU_FILE):
    print("Loading education dataset...")
    df_edu = pd.read_csv(EDU_FILE)
    print(f"Loaded {len(df_edu):,} education rows.")
    
    print("Left joining Education data on user_id...")
    df_events = pd.merge(df_events, df_edu, on="user_id", how="left")
    print(f"Shape after Education join: {df_events.shape}")
else:
    print("Warning: Education dataset not found. Skipping.")

Loading education dataset...


Loaded 987,658 education rows.
Left joining Education data on user_id...
Shape after Education join: (1239185, 50)


In [4]:
if os.path.exists(EXP_IND_FILE):
    print("Loading prior industry experience dataset...")
    df_exp_ind = pd.read_csv(EXP_IND_FILE)
    print(f"Loaded {len(df_exp_ind):,} industry experience rows.")
    
    print("Left joining Industry Experience data on user_id...")
    df_events = pd.merge(df_events, df_exp_ind, on="user_id", how="left")
    print(f"Shape after Industry Experience join: {df_events.shape}")
else:
    print("Warning: Industry experience dataset not found. Skipping.")

Loading prior industry experience dataset...


Loaded 803,184 industry experience rows.
Left joining Industry Experience data on user_id...


Shape after Industry Experience join: (1239185, 450)


In [5]:
if os.path.exists(EXP_SEN_FILE):
    print("Loading prior seniority experience dataset...")
    df_exp_sen = pd.read_csv(EXP_SEN_FILE)
    print(f"Loaded {len(df_exp_sen):,} seniority experience rows.")
    
    print("Left joining Seniority Experience data on user_id...")
    df_events = pd.merge(df_events, df_exp_sen, on="user_id", how="left")
    print(f"Shape after Seniority Experience join: {df_events.shape}")
else:
    print("Warning: Seniority experience dataset not found. Skipping.")

Loading prior seniority experience dataset...
Loaded 876,800 seniority experience rows.
Left joining Seniority Experience data on user_id...


Shape after Seniority Experience join: (1239185, 457)


In [6]:
if os.path.exists(EXP_ROLE_FILE):
    print("Loading prior role experience dataset...")
    df_exp_role = pd.read_csv(EXP_ROLE_FILE)
    print(f"Loaded {len(df_exp_role):,} role experience rows.")
    
    print("Left joining Role Experience data on user_id...")
    df_events = pd.merge(df_events, df_exp_role, on="user_id", how="left")
    # Fill NaN values for founders with no record in the role experience dataset
    df_events['exp_role_Business'] = df_events['exp_role_Business'].fillna(0.0)
    df_events['exp_role_Technical'] = df_events['exp_role_Technical'].fillna(0.0)
    print(f"Shape after Role Experience join: {df_events.shape}")
else:
    print("Warning: Role experience dataset not found. Skipping.")

Loading prior role experience dataset...
Loaded 681,262 role experience rows.
Left joining Role Experience data on user_id...


Shape after Role Experience join: (1239185, 459)


In [7]:
if os.path.exists(DETAILS_FILE):
    print("Loading prior position details dataset...")
    df_details = pd.read_csv(DETAILS_FILE)
    print(f"Loaded {len(df_details):,} prior position details rows.")
    
    print("Left joining Prior Position Details data on ['user_id', 'founding_event_number']...")
    df_events = pd.merge(df_events, df_details, on=["user_id", "founding_event_number"], how="left")
    print(f"Shape after Prior Position Details join: {df_events.shape}")
else:
    print("Warning: Prior position details dataset not found. Skipping.")

Loading prior position details dataset...


Loaded 1,239,185 prior position details rows.
Left joining Prior Position Details data on ['user_id', 'founding_event_number']...


Shape after Prior Position Details join: (1239185, 483)


In [8]:
if os.path.exists(PB_FILE):
    print("Loading Pitchbook venture outcomes dataset...")
    df_pb = pd.read_csv(PB_FILE)
    print(f"Loaded {len(df_pb):,} Pitchbook matched rows.")
    
    # Ensure keys are numeric for alignment
    df_events['venture_firm_id'] = pd.to_numeric(df_events['venture_firm_id'], errors='coerce')
    df_pb['rcid'] = pd.to_numeric(df_pb['rcid'], errors='coerce')
    
    print("Left joining Pitchbook data on venture_firm_id = rcid...")
    df_events = pd.merge(
        df_events, 
        df_pb, 
        left_on="venture_firm_id", 
        right_on="rcid", 
        how="left"
    )
    # Drop the redundant rcid column
    if 'rcid' in df_events.columns:
        df_events = df_events.drop(columns=['rcid'])
        
    print(f"Shape after Pitchbook join: {df_events.shape}")
else:
    print("Warning: Pitchbook dataset not found. Skipping.")

Loading Pitchbook venture outcomes dataset...
Loaded 36,129 Pitchbook matched rows.
Left joining Pitchbook data on venture_firm_id = rcid...


Shape after Pitchbook join: (1239185, 495)


In [9]:
print(f"Saving final combined founder-level dataset to: {FINAL_OUTPUT_FILE}")
df_events.to_csv(FINAL_OUTPUT_FILE, index=False)
print("Combination process successfully completed!")

Saving final combined founder-level dataset to: /Users/milanmiric/Documents/Research/Active Papers/LinkedIn Entreprenurship & Experience [Paper 2]/Data Extraction and Analysis/D - Data/D3 - Refined Datasets for Analysis/Founder_Level_Merged_Dataset.csv


Combination process successfully completed!
